In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Example paths to your dataset
verilog_file = "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.src"
c_code_file = "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.tgt"
verilog_test_file = '/content/drive/MyDrive/BTP/Consolidated_data_final_1375/test.src'
c_code_test_file = '/content/drive/MyDrive/BTP/Consolidated_data_final_1375/test.tgt'

In [ ]:
from transformers import T5ForConditionalGeneration, RobertaTokenizer

# Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')
model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small', device_map="auto")

# File paths
verilog_file = "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.src"
c_code_file = "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.tgt"

# Function to calculate the max length (number of tokens)
def calculate_max_length(file_path, tokenizer):
    max_length = 0
    with open(file_path, 'r') as file:
        lines = file.readlines()
        for line in lines:
            # Tokenize the line
            tokens = tokenizer(line.strip(), return_tensors="pt").input_ids
            # Update max_length if the current line has more tokens
            max_length = max(max_length, tokens.shape[1])
    return max_length

# Calculate max length for Verilog and C code files
max_length_verilog = calculate_max_length(verilog_file, tokenizer)
max_length_c_code = calculate_max_length(c_code_file, tokenizer)

print(f"Max length for Verilog file: {max_length_verilog}")
print(f"Max length for C code file: {max_length_c_code}")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (6422 > 512). Running this sequence through the model will result in indexing errors


Max length for Verilog file: 266372
Max length for C code file: 798


In [ ]:
from transformers import RobertaTokenizer
import torch

def find_max_token_length(file_path, tokenizer):
    max_length = 0
    current_snippet = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()

            if line:
                current_snippet.append(line)
            else:
                # Empty line indicates the end of a Verilog code snippet
                if current_snippet:
                    snippet_text = '\n'.join(current_snippet)
                    tokens = tokenizer.encode(snippet_text, add_special_tokens=True)
                    max_length = max(max_length, len(tokens))
                    current_snippet = []

    # Check for the last snippet
    if current_snippet:
        snippet_text = '\n'.join(current_snippet)
        tokens = tokenizer.encode(snippet_text, add_special_tokens=True)
        max_length = max(max_length, len(tokens))

    return max_length

# Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')

# Usage
file_path =  "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.src"
max_token_length = find_max_token_length(file_path, tokenizer)

# Suggest a max_length value (e.g., next power of 2, with some padding)
suggested_max_length = 2 ** (max_token_length - 1).bit_length()

print(f"Maximum token length across all Verilog code snippets: {max_token_length}")
print(f"Suggested max_length for T5 model: {suggested_max_length}")

# Example of how to use this with your model
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small', device_map="auto")

# Use the suggested max_length in your model operations
# For example, when generating:
# outputs = model.generate(input_ids, max_length=suggested_max_length)

Token indices sequence length is longer than the specified maximum sequence length for this model (11262396 > 512). Running this sequence through the model will result in indexing errors


Maximum token length across all Verilog code snippets: 11262396
Suggested max_length for T5 model: 16777216


In [ ]:
from transformers import RobertaTokenizer
import torch

def find_max_token_length(file_path, tokenizer):
    max_length = 0

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()

            if line:
                # Tokenize each line individually since there are no empty lines
                tokens = tokenizer.encode(line, add_special_tokens=True)
                max_length = max(max_length, len(tokens))

    return max_length

# Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')

# Usage
file_path = "/content/drive/MyDrive/BTP/Consolidated_data_final_1375/train.src"
max_token_length = find_max_token_length(file_path, tokenizer)

# Suggest a max_length value (e.g., next power of 2, with some padding)
suggested_max_length = 2 ** (max_token_length - 1).bit_length()

print(f"Maximum token length across all Verilog code lines: {max_token_length}")
print(f"Suggested max_length for T5 model: {suggested_max_length}")

# Example of how to use this with your model
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small', device_map="auto")

# Use the suggested max_length in your model operations
# For example, when generating:
# outputs = model.generate(input_ids, max_length=suggested_max_length)


Token indices sequence length is longer than the specified maximum sequence length for this model (6422 > 512). Running this sequence through the model will result in indexing errors


Maximum token length across all Verilog code lines: 266372
Suggested max_length for T5 model: 524288
